# 🧠 Exercise: Build a Memory Store That Remembers Users
**O'Reilly Live Training — AI Agent Memory Essentials**  
**Segment 2: Vector Memory with ChromaDB**

---

## What you're building in 10 minutes

A memory pipeline that:
1. **Stores** user preferences into ChromaDB
2. **Retrieves** relevant memories when the user sends a new message
3. **Injects** those memories into a system prompt
4. **Calls** an LLM that responds as if it genuinely knows the user

By the end, your agent will go from **goldfish** (forgets everything) to **personal trainer** (remembers everything that matters).

---

## Setup check
Run this cell first. If it passes, you're ready to go. ✅

In [15]:
# ✅ Setup check — run this first
!pip install chromadb openai
try:
    import chromadb
    from openai import OpenAI
    print("✅ chromadb imported successfully — version:", chromadb.__version__)
    print("✅ openai imported successfully")
    print("\n🚀 You're ready to build!")
except ImportError as e:
    print(f"❌ Missing package: {e}")
    print("Run: pip install chromadb openai")

✅ chromadb imported successfully — version: 1.5.9
✅ openai imported successfully

🚀 You're ready to build!


---
## Step 1 — Initialize ChromaDB and Create a Collection

Think of this as **building the filing cabinet** and labeling the first folder.

We're using an **in-memory** client so nothing needs to be installed or configured — it just works locally.

In [16]:
import chromadb

# In-memory client — no setup needed, perfect for the exercise
# For production you'd use: chromadb.PersistentClient(path="./chroma_store")
client = chromadb.Client()

# Create (or get) a collection — this is our "user_preferences" folder
collection = client.get_or_create_collection(
    name="user_preferences"
)

print("✅ Collection created:", collection.name)
print("📂 Filing cabinet is open and ready.")

✅ Collection created: user_preferences
📂 Filing cabinet is open and ready.


---
## Step 2 — Store Memories for a User

We're filing **preference cards** for user `u42`.

Each card has:
- A **document** (the actual memory text)
- An **id** (unique name for the card)
- **Metadata** (the label — who it belongs to, when it was stored)

ChromaDB auto-embeds the text using its built-in model. No OpenAI call needed for storage.

In [17]:
# User preferences to store — 5 memory cards for user u42
memories = [
    "User prefers dark mode across all interfaces",
    "User wants concise answers, no long explanations",
    "User is a drone operator based in Charlotte NC",
    "User gets frustrated when responses exceed 3 paragraphs",
    "User prefers bullet points over prose for technical topics"
]

ids = [f"mem_{i:03d}" for i in range(len(memories))]

metadatas = [
    {"user_id": "u42", "category": "ui_preference", "ts": "2026-06"},
    {"user_id": "u42", "category": "communication", "ts": "2026-06"},
    {"user_id": "u42", "category": "profile",       "ts": "2026-06"},
    {"user_id": "u42", "category": "communication", "ts": "2026-06"},
    {"user_id": "u42", "category": "communication", "ts": "2026-06"},
]

# Store all 5 cards in one call
collection.add(
    documents=memories,
    ids=ids,
    metadatas=metadatas
)

print(f"✅ Stored {len(memories)} memories for user u42")
print("📇 Cards filed:", ids)

✅ Stored 5 memories for user u42
📇 Cards filed: ['mem_000', 'mem_001', 'mem_002', 'mem_003', 'mem_004']


---
## Step 3 — Retrieve Relevant Memories

Now a new message comes in from u42.

We query ChromaDB with that message — it finds the **most semantically relevant** cards.

⚠️ Notice: the query words don't exactly match the stored words. That's the point — **meaning over keywords**.

In [18]:
# New message from the user
user_message = "How should I configure my dashboard settings?"
user_id = "u42"

# Query ChromaDB — semantic match, not keyword match
results = collection.query(
    query_texts=[user_message],   # ChromaDB embeds this automatically
    n_results=3,                   # Return top 3 most relevant memories
    where={"user_id": user_id}    # Only search THIS user's cards
)

retrieved_memories = results["documents"][0]

print("🔍 Query:", user_message)
print("\n📋 Retrieved memories:")
for i, mem in enumerate(retrieved_memories, 1):
    print(f"  {i}. {mem}")

print("\n💡 Note: we searched for 'dashboard settings' and got back UI + communication preferences")
print("   That's semantic search — matching meaning, not keywords.")

🔍 Query: How should I configure my dashboard settings?

📋 Retrieved memories:
  1. User prefers dark mode across all interfaces
  2. User gets frustrated when responses exceed 3 paragraphs
  3. User wants concise answers, no long explanations

💡 Note: we searched for 'dashboard settings' and got back UI + communication preferences
   That's semantic search — matching meaning, not keywords.


---
## Step 4 — Inject Memories into the System Prompt

This is the **briefing handoff**.

Before the LLM sees the user's question — we slip in the memory cards as context.
The LLM reads the briefing first, then answers.

In [19]:
def build_prompt(user_msg: str, user_id: str) -> tuple[str, str]:
    """
    The full injection pipeline:
    1. Query ChromaDB for relevant memories
    2. Format them as context
    3. Inject into system prompt
    4. Return (system_prompt, user_message) ready for LLM
    """
    # Step 1: Retrieve
    results = collection.query(
        query_texts=[user_msg],
        n_results=3,
        where={"user_id": user_id}
    )
    memories = results["documents"][0]

    # Step 2: Format
    memory_block = "\n".join(f"- {m}" for m in memories)

    # Step 3: Inject
    system_prompt = f"""You are a helpful AI assistant.

User context from memory:
{memory_block}

Use this context to personalize your response. Do not mention that you have memory — just respond naturally."""

    return system_prompt, user_msg


# Preview what gets sent to the LLM
system_prompt, msg = build_prompt(user_message, user_id)

print("📨 SYSTEM PROMPT (what the LLM reads first):")
print("-" * 50)
print(system_prompt)
print("-" * 50)
print("\n💬 USER MESSAGE:", msg)

📨 SYSTEM PROMPT (what the LLM reads first):
--------------------------------------------------
You are a helpful AI assistant.

User context from memory:
- User prefers dark mode across all interfaces
- User gets frustrated when responses exceed 3 paragraphs
- User wants concise answers, no long explanations

Use this context to personalize your response. Do not mention that you have memory — just respond naturally.
--------------------------------------------------

💬 USER MESSAGE: How should I configure my dashboard settings?


---
## Step 5 — Call the LLM and See the Magic ✨

Now we send both to the LLM.

The LLM was NOT trained on this user's preferences. It has no memory of its own.

But watch how it responds — **as if it genuinely knows u42**.

In [20]:
import os
from openai import OpenAI
from google.colab import userdata

# ✅ Reads from Colab Secrets — key never visible on screen
# To set up: click the 🔑 key icon in the left sidebar → Add secret → OPENAI_API_KEY
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client_llm = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def ask_agent(user_msg: str, user_id: str) -> str:
    system_prompt, msg = build_prompt(user_msg, user_id)

    response = client_llm.chat.completions.create(
        model="gpt-4o-mini",  # cheap and fast — perfect for live demo
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": msg}
        ],
        max_tokens=300
    )
    return response.choices[0].message.content


# Run it!
print("💬 User:", user_message)
print("\n Agent (with memory):")
print("-" * 50)
response = ask_agent(user_message, user_id)
print(response)

💬 User: How should I configure my dashboard settings?

 Agent (with memory):
--------------------------------------------------
To configure your dashboard settings effectively:

1. **Choose a Layout**: Opt for a layout that prioritizes important metrics. Grid or list views can be efficient based on your data needs.

2. **Widgets and Alerts**: Add relevant widgets that showcase key performance indicators (KPIs). Set up alerts for metrics that require immediate attention.

3. **Dark Mode**: Make sure to enable dark mode for better visibility and to reduce eye strain, especially during long sessions.

Adjust these settings periodically based on your usage and changing priorities.


---
## 🆚 Bonus: See the Difference — With vs Without Memory

Run this cell to see the **exact same question** answered two ways:
- 🐟 **Without memory** — generic goldfish response
- 🧠 **With memory** — personalized personal trainer response

In [ ]:
def ask_agent_no_memory(user_msg: str) -> str:
    """Goldfish mode — no memory, bare system prompt"""
    response = client_llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful AI assistant."},
            {"role": "user",   "content": user_msg}
        ],
        max_tokens=300
    )
    return response.choices[0].message.content


test_question = "How should I configure my dashboard settings?"

print(" WITHOUT MEMORY (goldfish):")
print("-" * 50)
print(ask_agent_no_memory(test_question))

print("\n" + "=" * 50 + "\n")

print(" WITH MEMORY (personal trainer):")
print("-" * 50)
print(ask_agent(test_question, "u42"))

print("\n\n Same question. Same LLM. Completely different response.")
print("   The only difference: we briefed the trainer before they walked in.")

 WITHOUT MEMORY (goldfish):
--------------------------------------------------
Configuring your dashboard settings effectively depends on several factors, including your specific goals, the platform you're using, and the type of data you want to visualize. Here are some general steps to consider when setting up your dashboard:

1. **Define Your Goals**:
   - Understand what you want to achieve with the dashboard. Are you tracking performance, monitoring KPIs, visualizing data trends, or reporting findings?

2. **Identify Key Metrics**:
   - Select the metrics that are most important for your goals. This could include sales numbers, user engagement, financial performance, etc.

3. **Choose Visualization Types**:
   - Decide how to best represent your data. Common visualization types include:
     - Line charts for trends over time.
     - Bar charts for comparisons.
     - Pie charts for proportions.
     - Tables for detailed data.
   
4. **Organize Layout and Design**:
   - Arrange wi

---
## 🎯 Challenge: Add Your Own Memories

Try these before the segment ends:

**Challenge 1 — Add a new memory card:**
```python
collection.add(
    documents=["User is preparing for a FAA security audit in July"],
    ids=["mem_005"],
    metadatas=[{"user_id": "u42", "category": "context", "ts": "2026-06"}]
)
```
Then ask: *"What should I prioritize this month?"* — does the response change?

**Challenge 2 — Try a second user:**  
Store memories for `user_id: "u99"` with completely different preferences.  
Ask the same question for both users. Watch the agent respond differently to each.

**Challenge 3 — Break the keyword assumption:**  
Store: *"User dislikes verbose explanations"*  
Query: *"How do you communicate?"*  
ChromaDB should still find it. Meaning over keywords. ✅

---

## ✅ What you just built

| Step | What happened |
|------|---------------|
| `collection.add()` | Filed memory cards with labels |
| `collection.query()` | Found relevant cards by meaning, not keywords |
| `build_prompt()` | Briefed the LLM before the conversation started |
| `ask_agent()` | Full pipeline: retrieve → inject → respond |

**This is the foundation of every production memory layer.**  
Segment 3 adds conversation patterns on top of this.  
Segment 4 replaces the manual wiring with Mem0 and Zep.  

But the core loop? Always the same three steps: **embed → persist → query.** 🎯

In [ ]:
# -*- coding: utf-8 -*-
"""
O'Reilly Live Training — AI Agent Memory Essentials
Segment 2: Vector Memory with ChromaDB
🎯 Challenge Solutions


"""

# ============================================================
# Challenge 1 — Add a new memory card and see if it changes the response
# ============================================================

collection.add(
    documents=["User is preparing for a FAA security audit in July"],
    ids=["mem_005"],
    metadatas=[{"user_id": "u42", "category": "context", "ts": "2026-06"}]
)

print("✅ Challenge 1 — new memory card filed for u42")

challenge_1_question = "What should I prioritize this month?"

print("\n💬 User:", challenge_1_question)
print("\n Agent (before vs after mem_005):")
print("-" * 50)

# Before: temporarily query without the new card to show contrast
before_results = collection.query(
    query_texts=[challenge_1_question],
    n_results=3,
    where={
        "$and": [
            {"user_id": {"$eq": "u42"}},
            {"category": {"$ne": "context"}}  # exclude the new card
        ]
    }
)
print("Retrieved WITHOUT audit memory:", before_results["documents"][0])

after_response = ask_agent(challenge_1_question, "u42")
print("\nRetrieved for the live agent call now naturally includes the audit context.")
print("\n🤖 Full response:")
print(after_response)

print("\n💡 Notice: 'prioritize this month' semantically matches the audit-prep")
print("   memory even though neither shares an exact keyword with it.")




In [ ]:

# ============================================================
# Challenge 2 — Store memories for a second user, u99, and compare
# ============================================================

memories_u99 = [
    "User prefers light mode and high-contrast UI",
    "User wants detailed, thorough explanations with examples",
    "User is a product manager based in Austin TX",
    "User likes step-by-step walkthroughs, not summaries",
    "User is onboarding a new team this quarter"
]

ids_u99 = [f"mem_u99_{i:03d}" for i in range(len(memories_u99))]

metadatas_u99 = [
    {"user_id": "u99", "category": "ui_preference", "ts": "2026-06"},
    {"user_id": "u99", "category": "communication", "ts": "2026-06"},
    {"user_id": "u99", "category": "profile",       "ts": "2026-06"},
    {"user_id": "u99", "category": "communication", "ts": "2026-06"},
    {"user_id": "u99", "category": "context",       "ts": "2026-06"},
]

collection.add(
    documents=memories_u99,
    ids=ids_u99,
    metadatas=metadatas_u99
)

print("\n\n✅ Challenge 2 — stored 5 memories for user u99")

shared_question = "How should I configure my dashboard settings?"

print(f"\n💬 Same question for both users: \"{shared_question}\"")

print("\n u42 (dark mode, concise, drone ops):")
print("-" * 50)
print(ask_agent(shared_question, "u42"))

print("\n u99 (light mode, detailed, PM):")
print("-" * 50)
print(ask_agent(shared_question, "u99"))

print("\n Same question, same model, same pipeline — completely different")
print("   answers because the `where={'user_id': ...}` filter scopes retrieval")
print("   to each user's own memory cards.")

In [ ]:

# ============================================================
# Challenge 3 — Break the keyword assumption (semantic vs. literal match)
# ============================================================

collection.add(
    documents=["User dislikes verbose explanations"],
    ids=["mem_006"],
    metadatas=[{"user_id": "u42", "category": "communication", "ts": "2026-06"}]
)

print("\n\n✅ Challenge 3 — stored 'User dislikes verbose explanations' for u42")

keyword_test_query = "How do you communicate?"

results_c3 = collection.query(
    query_texts=[keyword_test_query],
    n_results=3,
    where={"user_id": "u42"}
)

print(f"\n🔍 Query: \"{keyword_test_query}\"")
print("📋 Retrieved memories:")
for i, mem in enumerate(results_c3["documents"][0], 1):
    print(f"  {i}. {mem}")

found = any("verbose" in m.lower() for m in results_c3["documents"][0])
print(f"\n{'✅' if found else '❌'} 'dislikes verbose explanations' retrieved: {found}")
print("💡 Zero shared keywords between query and document — pure semantic match.")


# ============================================================
# Recap
# ============================================================
print("\n" + "=" * 50)
print("🎯 Challenges complete:")
print("  1. New memory card → agent response updates without code changes")
print("  2. Second user (u99) → same pipeline, isolated per-user memory")
print("  3. Zero-keyword-overlap query → still retrieved correctly")
print("=" * 50)